## Decision Tree Regression

## Import Libraries

In [ ]:
import pandas as pd # Used for Loading and working with tabular data
import numpy as np # Used for numerical calculations

import matplotlib.pyplot as plt # Used for visualization

from sklearn.model_selection import train_test_split # Used to split data into train/test sets
from sklearn.compose import ColumnTransformer # Used to apply preprocessing to selected column types
from sklearn.preprocessing import OneHotEncoder # Used to encode categorical variables
from sklearn.pipeline import Pipeline # Used to combine preprocessing and model steps

from sklearn.tree import DecisionTreeRegressor, plot_tree # Main tree regression model and tree visualization 

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score # Regression evaluation metrics 





## 3. Load Data

In [ ]:
df = pd.read_csv("../data/processed/member_analysis_ready.csv") # Load the feature-engineered dataset

df.head() # Preview the first few rows


## 4. Define Target and Features

In [ ]:
target = "monthly_cost" # Define the regression target 

drop_cols = [
    "member_id", # Identifier column; not meaningful for prediction
    "monthly_cost", # Target column; must be removed from features
    "high_cost_member", # Derived from monthly_cost, so it leaks target information
    "awv_completed" # Excluded to keep cost model focused on risk, access, and utilization
]

X = df.drop(columns = drop_cols) # Create feature matrix without leakage columns
y = df[target] # Create target vector

## 5. Identify Categorical and Numeric Columns

In [ ]:
categorical_cols = X.select_dtypes(
    include = ["object", "string", "category", "bool"]
).columns.tolist() # Detect categorical columns safely

numeric_cols = X.select_dtypes(
    include = ["int64", "float64", "int32", "float32"] 
).columns.tolist()

print("Categorical columns:", categorical_cols)
print("Numeric columns:", numeric_cols)


## 6. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, # Feature matrix
    y, # Target Vector
    test_size = 0.20, # Use 20% of data for testing
    random_state = 42  # Make split reproducible 
)

## 7. Build Baseline Decision Tree Pipeline

In [ ]:
preprocessor = ColumnTransformer(
    transformer=[
        (
            "cat",
            OneHotEncoder(handle_unknown = "ignore"),
            categorical_cols 
        )
    ],
    remainder = "passthrough" # Keep numeric columns unchanged 
)


tree_model = Pipeline(
    steps = [
        ("preprocessor", preprocessor), # Convert categorical variables to numeric format
        (
            "model",
            DecisionTreeRegressor(
                random_state= = 42
            )
        )
    ]
)



## 8. Fit the Model

In [ ]:
tree_model.fit(X_train, y_train) # Train the decision tree model 

## 9. Evaluate Train vs Test Performance

In [ ]:
y_train_pred = tree_model.predict(X_train) # Predict on training data
y_test_pred = tree_model.predict(X_test) # Predict on test data

train_mae = mean_absolute_error(y_train, y_train_pred)
test_mae = mean_absolute_error(y_test, y_test_pred)

train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

metrics_df = pd.DataFrame({
    "Dataset": ["Train", "Test"],
    "MAE": [train_mae, test_mae],
    "RMSE": [train_rmse, test_rmse],
    "R2": [train_r2, test_r2]
})

metrics_df





## 10. Controlled Tree With Max Depth

In [ ]:
controlled_tree_model = Pipeline(
    steps = [
        ("preprocessor", preprocessor),
        (
                "model",
                DecisionTreeRegressor(
                    max_depth = 5, # Limit tree complexity
                    min_samples_leaf=25, # Require enough members in each final leaf 
                    random_state=42 
                )
        )
    ]
)

controlled_tree_model.fit(X_train, y_train) # Train controlled tree